# Edge AI Deployment Tutorial

Build a voice-enabled AI assistant that runs entirely offline on edge devices.

## System Architecture

```mermaid
flowchart LR
    A[User Input] --> B[Model Selector]
    B --> C{Query Type}
    C -->|Simple| D[Local Qwen3]
    C -->|Complex| E[Cloud Model]
    D --> F[Agent Orchestrator]
    E --> F
    F --> G[Vehicle Agents]
    G --> H[Virtual ECU]
    H --> I[CAN Bus]
```

## Components

- **Model**: Qwen3-1.7B (4.7GB quantized)
- **Voice**: FFmpeg with integrated Whisper
- **Agents**: 5 vehicle control agents (climate, windows, seats, lights, drive mode)
- **Backend**: Virtual ECU with safety validation

## Deployment Sequence

```mermaid
sequenceDiagram
    participant User
    participant Setup
    participant Docker
    participant Models
    participant Server
    
    User->>Setup: ./setup.sh
    Setup->>Docker: Build container
    Docker->>Models: Download Qwen3 + Whisper
    Models->>Server: Start llama-server
    Server->>User: Ready on :8080
```

## Step 1: Environment Configuration

Configure the runtime environment with optimal settings for edge deployment. This step sets critical parameters like context window size, thread count, and memory limits. The configuration adapts automatically based on available hardware resources. These settings directly impact model performance and memory usage on edge devices.

In [ ]:
import os
import subprocess
import requests
import json
import time
from pathlib import Path

project_root = Path("/Users/aaronbw/Documents/DEV/v1/DEV/NEW/samples-dev/02-samples/14-agentic-ai-at-the-edge")
os.chdir(project_root)

env_config = {
    "LLAMACPP_URL": "http://localhost:8080",
    "LLAMA_CTX_SIZE": "2048",
    "GGML_NTHREADS": "4",
    "MAX_TOKENS": "1024",
    "USE_RICH_UI": "false",
    "DRIVER_PROFILE": "guest"
}

for key, value in env_config.items():
    os.environ[key] = value

print(f"Configured: {env_config['LLAMA_CTX_SIZE']} context, {env_config['GGML_NTHREADS']} threads")

## Step 2: Build Container

Build or start the Docker container that hosts the edge AI system. The container includes FFmpeg with Whisper for voice processing, llama.cpp for model inference, and all Python dependencies. If the container already exists, it checks the status and starts it if needed. The build process compiles optimized binaries for the target architecture (x86/ARM64).

In [ ]:
deployment_path = project_root / "src/edge/deployment"

result = subprocess.run(
    ["docker", "ps", "-a", "--filter", "name=strands-edge-personal-assistant", "--format", "{{.Names}}"],
    capture_output=True,
    text=True
)

if "strands-edge-personal-assistant" in result.stdout:
    status = subprocess.run(
        ["docker", "inspect", "strands-edge-personal-assistant", "--format", "{{.State.Running}}"],
        capture_output=True,
        text=True
    )
    if "true" in status.stdout:
        print("Container running")
    else:
        subprocess.run(["docker", "start", "strands-edge-personal-assistant"])
        print("Container started")
else:
    print("Building container...")
    subprocess.run([str(deployment_path / "setup.sh")], cwd=deployment_path)
    print("Container ready")

## Step 3: Test Component Flow

Validate the complete data flow from user input through model selection to ECU execution. This demonstrates how the system routes commands through the appropriate model based on complexity analysis. The local model handles vehicle control commands for low latency while complex queries can be routed to cloud models. The virtual ECU validates all commands for safety before execution.

In [ ]:
import sys
sys.path.insert(0, str(project_root))

from main import process_input
from src.data.vehicle_systems import get_virtual_ecu
from src.agents.tools.model_selector import select_model

test_command = "Set temperature to 72"

# Model selection
selection = select_model(test_command)
print(f"Model: {selection['provider']}")

# Process command
response = process_input(test_command)
print(f"Response: {response}")

# Check ECU state
ecu = get_virtual_ecu()
climate = ecu.get_state("climate")
print(f"ECU: {climate['temperature_set']}°F")

## Step 4: Voice Processing

Validate FFmpeg's Whisper integration for speech-to-text capabilities. The system adapts to different platforms automatically - Linux uses direct audio device access, macOS uses file-based exchange due to Docker limitations, and API mode uses Base64-encoded audio. This abstraction ensures voice input works consistently across all deployment scenarios.

| Platform | Audio Method |
|----------|-------------|
| Linux | Direct `/dev/snd` |
| macOS | File exchange |
| Container | Volume mount |
| API | Base64 JSON |

In [ ]:
import platform

ffmpeg_path = os.environ.get("FFMPEG_PATH", "ffmpeg")
result = subprocess.run([ffmpeg_path, "-hide_banner", "-filters"], capture_output=True, text=True)

if "whisper" in result.stdout:
    system = platform.system()
    if system == "Darwin":
        cmd = "avfoundation audio"
    else:
        cmd = "pulse audio"
    print(f"Whisper available ({cmd})")
else:
    print("Whisper not available")

## Step 5: Safety System

Demonstrate the virtual ECU's safety validation that prevents dangerous operations. The ECU enforces real-world constraints like preventing seat adjustments while driving or limiting window operations at high speeds. Every command passes through this safety layer before reaching the simulated CAN bus. This architecture mirrors actual automotive safety systems used in production vehicles.

| Component | Speed Limit | Range |
|-----------|------------|-------|
| Seats | 0 mph | 0-100% |
| Windows | <45 mph | 0-100% |
| Drive Mode | <5 mph | Valid modes |
| Climate | None | 60-85°F |
| Lighting | None | Mode-specific |

In [ ]:
from src.data.vehicle_systems import get_virtual_ecu

ecu = get_virtual_ecu("TechCar Model X")

# Safe command
result = ecu.execute_command({"component": "climate", "action": "set_temperature", "value": 72})
print(f"Safe: {result['success']}")

# Unsafe command
result = ecu.execute_command({"component": "climate", "action": "set_temperature", "value": 95})
print(f"Unsafe: {result['success']} - {result['message']}")

# Speed check
ecu.simulate_driving(speed=30, gear="D")
result = ecu.execute_command({"component": "seats", "action": "adjust_position", "seat": "driver", "forward": 60})
print(f"Seat at 30mph: {result['success']}")
ecu.simulate_driving(speed=0, gear="P")

## Step 6: Model Selection

Test the intelligent model routing that optimizes for latency and accuracy. The local Qwen3 model analyzes each query to determine if it can handle it locally or needs cloud assistance. Vehicle control commands always stay local for safety and latency reasons. This hybrid approach enables both offline operation and access to powerful cloud models when needed.

**Local Model**: Vehicle controls, real-time needs, offline operation

**Cloud Model**: Creative writing, complex analysis, large code generation

In [ ]:
from src.agents.tools.model_selector import select_model

queries = [
    "Turn on AC",
    "What time is it?",
    "Write 500-word essay"
]

for q in queries:
    s = select_model(q)
    print(f"{q:20} -> {s['provider']}")

## Step 7: Deployment Modes

Detect the current deployment mode to adapt behavior accordingly. Development mode runs directly on the host with full resources, container mode runs isolated with resource limits, and API mode exposes HTTP endpoints for integration. The same codebase adapts to all three modes automatically. This unified architecture simplifies deployment across different environments.

| Mode | Command | Resources |
|------|---------|----------|
| Development | `python main.py` | Unlimited |
| Container | `docker run -d edge` | 4GB/2CPU |
| API | `ENABLE_API=true docker run` | Scalable |

In [ ]:
in_docker = os.path.exists("/.dockerenv")
api_mode = os.getenv("ENABLE_API", "false") == "true"

if in_docker:
    mode = "API" if api_mode else "Container"
else:
    mode = "Development"

print(f"Mode: {mode}")

## Step 8: Performance Testing

Measure end-to-end latency for typical vehicle control commands. The system achieves sub-500ms response times for most operations through optimizations like model quantization, context caching, and efficient tool routing. These benchmarks validate that the system meets automotive real-time requirements. Performance scales linearly with available CPU cores up to 4 threads.

In [ ]:
import time
import statistics

commands = [
    "Turn on lights",
    "Set temperature to 70",
    "Close all windows"
]

latencies = []
for cmd in commands:
    start = time.time()
    process_input(cmd)
    latencies.append((time.time() - start) * 1000)

print(f"Avg: {statistics.mean(latencies):.0f}ms, Min: {min(latencies):.0f}ms, Max: {max(latencies):.0f}ms")

## Summary

This tutorial demonstrated how to build and deploy a production-ready edge AI system that operates entirely offline. The architecture combines multiple advanced technologies into a unified solution that adapts seamlessly across development, container, and API deployment modes.
